In [1]:
import random
import numpy as np
import json

from torchvision import datasets
from torchvision import transforms

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

In [2]:
# Sanity check: verify class labels and number of images per split
train_ds = datasets.ImageFolder("hair-dataset/data/train")
eval_ds = datasets.ImageFolder("hair-dataset/data/test")

print(train_ds.class_to_idx)          # {'curly': 0, 'straight': 1}
print(len(train_ds), len(eval_ds))    # 800 201

{'curly': 0, 'straight': 1}
800 201


##### Reproducibility rules

In [3]:
# Reproducibility: fix all random seeds so every run gives the same results
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Force deterministic algorithms (no run-to-run variation)
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.deterministic = True   # only relevant on GPU
torch.backends.cudnn.benchmark = False      # don't auto-pick fastest (non-deterministic) algorithms
torch.set_num_threads(2)                    # same number of CPU threads as the reference script

# Train on CPU, as required for the reference results
device = torch.device("cpu")

##### Model and preprocessing

In [4]:
# Preprocessing: resize to 200x200, convert to tensor, normalize with ImageNet stats
# Used for both training (baseline, no augmentation) and evaluation
evaluation_transform = transforms.Compose([
    transforms.Resize((200, 200), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),                      # PIL image -> tensor with values in [0, 1]
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],             # per-channel mean (R, G, B)
        std=[0.229, 0.224, 0.225],              # per-channel std (R, G, B)
    ),
])

# Load the fixed train and held-out evaluation splits (labels: curly=0, straight=1)
train_dataset = datasets.ImageFolder("hair-dataset/data/train", transform=evaluation_transform)
evaluation_dataset = datasets.ImageFolder("hair-dataset/data/test", transform=evaluation_transform)

In [5]:
# Fixed generator so the training data is shuffled the same way every run
loader_generator = torch.Generator().manual_seed(SEED)

# Training loader: shuffled in batches of 20, single process for reproducibility
train_loader = DataLoader(
    train_dataset,
    batch_size=20,
    shuffle=True,
    num_workers=0,
    generator=loader_generator,
)

# Evaluation loader: no shuffling needed
evaluation_loader = DataLoader(
    evaluation_dataset,
    batch_size=20,
    shuffle=False,
    num_workers=0,
)

In [6]:
class HairModel(nn.Module):
    def __init__(self):
        super().__init__()
        # Convolution: 3 input channels (RGB) -> 32 feature maps, 3x3 filter
        # Output size: 200 - 3 + 1 = 198 -> 32 x 198 x 198
        self.conv = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=0)
        # Max pooling halves the size: 198 -> 99 -> 32 x 99 x 99
        self.pool = nn.MaxPool2d(kernel_size=2)
        # Fully connected layer: flattened features -> 64 neurons
        self.hidden = nn.Linear(32 * 99 * 99, 64)
        # Output layer: 1 value (logit) for binary classification
        self.output = nn.Linear(64, 1)

    def forward(self, x):
        x = self.pool(nn.functional.relu(self.conv(x)))  # conv -> ReLU -> pool
        x = x.flatten(start_dim=1)                        # flatten to (batch, features)
        x = nn.functional.relu(self.hidden(x))            # hidden layer + ReLU
        return self.output(x)                             # raw logits, no sigmoid here

# Create the model after the seeds so the initial weights are reproducible
model = HairModel().to(device)

In [7]:
# Binary cross-entropy loss that applies sigmoid internally (expects logits)
criterion = nn.BCEWithLogitsLoss()

# Stochastic gradient descent with momentum
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.002,        # learning rate
    momentum=0.8,    # smooths updates using previous gradients
)

##### Question 1 — loss and output contract

In [8]:
# Take the first 20 evaluation images directly from the dataset (doesn't use the random generator)
images = torch.stack([evaluation_dataset[i][0] for i in range(20)])
labels = torch.tensor(evaluation_dataset.targets[:20])
with torch.no_grad():                       # no gradients needed, just a check
    logits = model(images)                  # raw model output
y = labels.float().unsqueeze(1)             # labels as float, shape [20, 1] (same as output)

# Claim 1: one raw logit per image
print("Output shape:", logits.shape)
print("Sigmoid in model:", any(isinstance(m, nn.Sigmoid) for m in model.modules()))  # no sigmoid, so raw logits

# Claim 2: the loss = sigmoid + binary cross-entropy
p = torch.sigmoid(logits)                   # logits -> probabilities (0 to 1)
print("BCEWithLogitsLoss:", nn.BCEWithLogitsLoss()(logits, y).item())       # built-in: sigmoid happens inside the loss
print("Sigmoid + BCE by hand:", -(y * p.log() + (1 - y) * (1 - p).log()).mean().item())  # BCE formula -> same value

# Claim 3: the loss is numerically stable
x, t = torch.tensor([[-100.0]]), torch.tensor([[1.0]])                     # very confident 'curly', true label 'straight'
print("Naive:", -(t * torch.sigmoid(x).log()).item(),                      # sigmoid(-100) -> 0, log(0) -> inf
      "| Stable:", nn.BCEWithLogitsLoss()(x, t).item())                     # rewritten formula -> correct loss of 100

Output shape: torch.Size([20, 1])
Sigmoid in model: False
BCEWithLogitsLoss: 0.6279961466789246
Sigmoid + BCE by hand: 0.6279962658882141
Naive: inf | Stable: 100.0


In [9]:
# Why the other three losses don't fit

# CrossEntropyLoss: expects one logit PER CLASS (2 outputs) + softmax; we have 1 output -> only class 0 exists
try: nn.CrossEntropyLoss()(logits, torch.ones(20, dtype=torch.long))
except Exception as e: print("CrossEntropyLoss:", e)                               # label 1 -> out of bounds

# MSELoss: regression loss, just (prediction - label)^2 -> (10 - 1)^2 = 81
print("MSELoss (logit 10, label 1):", nn.MSELoss()(torch.tensor([[10.0]]), t).item())  # punishes a confident correct prediction

# CosineEmbeddingLoss: compares TWO vectors (input1, input2, target 1/-1); we only have one output + class label
try: nn.CosineEmbeddingLoss()(logits, y)
except Exception as e: print("CosineEmbeddingLoss:", e)                            # missing argument -> not a classification loss

CrossEntropyLoss: Target 1 is out of bounds.
MSELoss (logit 10, label 1): 81.0
CosineEmbeddingLoss: CosineEmbeddingLoss.forward() missing 1 required positional argument: 'target'


**Explanation:** The model outputs one raw logit per image (no sigmoid).
`BCEWithLogitsLoss` applies sigmoid + BCE in a stable way. The other three losses
don't fit: they need 2 outputs, do regression, or compare embeddings.

##### Question 2 - parameter count

In [10]:
# Total number of trainable parameters (weights + biases)
print(sum(p.numel() for p in model.parameters()))

20073473


**Calculation:** 
- Conv2d: 3·3·3·32 + 32 = 896
- Hidden Linear: 313,632·64 + 64 = 20,072,512
- Output Linear: 64·1 + 1 = 65

##### Training metrics

In [11]:
# Train the model for one epoch and return average loss and accuracy
def train_one_epoch(model, loader):
    model.train()                               # training mode (doesn't train yet, just sets the mode)
    loss_sum, correct = 0, 0                    
    for images, labels in loader:               # one batch of 20 images
        optimizer.zero_grad()                   # reset gradients from previous batch
        labels = labels.float().unsqueeze(1)    # labels to float with shape [20, 1], matching the logits
        logits = model(images)                  # forward prop: one logit per image
        loss = criterion(logits, labels)        # sigmoid + BCE
        loss.backward()                         # back prop: compute gradients
        optimizer.step()                        # update weights (Stochastic Gradient Descent)
        loss_sum += loss.item() * len(images)   # batch loss is an average -> multiply by batch size
        correct += ((torch.sigmoid(logits) >= 0.5) == labels).sum().item()  # prob >= 0.5 -> straight
    return loss_sum / len(loader.dataset), correct / len(loader.dataset)    # average per image

# Evaluate the model on held-out data (no learning) and return average loss and accuracy 
def evaluate(model, loader):
    model.eval()                                    # evaluation mode
    loss_sum, correct = 0, 0                        
    with torch.no_grad():                           # no gradients: only measure, don't learn
        for images, labels in loader:               
            labels = labels.float().unsqueeze(1)    
            logits = model(images)                 
            loss_sum += criterion(logits, labels).item() * len(images)       
            correct += ((torch.sigmoid(logits) >= 0.5) == labels).sum().item()  
    return loss_sum / len(loader.dataset), correct / len(loader.dataset)    

In [12]:
# Train for 10 epochs and track metrics per epoch
history = {"train_loss": [], "train_accuracy": [], "evaluation_loss": [], "evaluation_accuracy": []}

for epoch in range(10):
    train_loss, train_acc = train_one_epoch(model, train_loader)    # learn on training data
    eval_loss, eval_acc = evaluate(model, evaluation_loader)        # measure on held-out data

    history["train_loss"].append(train_loss)        # store results of this epoch
    history["train_accuracy"].append(train_acc)
    history["evaluation_loss"].append(eval_loss)
    history["evaluation_accuracy"].append(eval_acc)

    print(f"Epoch {epoch + 1}: train acc {train_acc:.3f}, eval acc {eval_acc:.3f}")

# Save the four series to JSON
with open("history_baseline.json", "w") as f:
    json.dump(history, f)

Epoch 1: train acc 0.645, eval acc 0.657
Epoch 2: train acc 0.718, eval acc 0.652
Epoch 3: train acc 0.755, eval acc 0.622
Epoch 4: train acc 0.791, eval acc 0.692
Epoch 5: train acc 0.809, eval acc 0.662
Epoch 6: train acc 0.807, eval acc 0.711
Epoch 7: train acc 0.836, eval acc 0.672
Epoch 8: train acc 0.885, eval acc 0.721
Epoch 9: train acc 0.910, eval acc 0.716
Epoch 10: train acc 0.939, eval acc 0.637


##### Question 3 — baseline accuracy

In [13]:
# Median of train_accuracy, rounded to 2 decimals
with open("history_baseline.json") as f:
    history_baseline = json.load(f)

print(round(np.median(history_baseline["train_accuracy"]), 2))

0.81


##### Question 4 — baseline loss

In [14]:
# Population standard deviation of train_loss, rounded to 3 decimals
print(round(np.std(history_baseline["train_loss"], ddof=0), 3))

0.139


##### Data augmentation

In [15]:
# Augmented transform: only for training, evaluation stays unchanged
train_transform_augmented = transforms.Compose([
    transforms.Resize((200, 200), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.RandomRotation(50, interpolation=transforms.InterpolationMode.NEAREST),  # rotate up to ±50°
    transforms.RandomResizedCrop(200, scale=(0.9, 1.0), ratio=(0.9, 1.1),
                                 interpolation=transforms.InterpolationMode.BILINEAR),  # small random crop + zoom
    transforms.RandomHorizontalFlip(),                                                  # mirror 50% of the time
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# New training dataset + loader with the augmented transform
train_dataset_augmented = datasets.ImageFolder("hair-dataset/data/train", transform=train_transform_augmented)
train_loader_augmented = DataLoader(
    train_dataset_augmented,
    batch_size=20,
    shuffle=True,
    num_workers=0,
    generator=torch.Generator().manual_seed(SEED + 1),   # new fixed generator (seed 43), same as the reference
)

In [16]:
# Continue training the SAME model and optimizer for 10 more epochs
history_aug = {"train_loss": [], "train_accuracy": [], "evaluation_loss": [], "evaluation_accuracy": []}

for epoch in range(10):
    train_loss, train_acc = train_one_epoch(model, train_loader_augmented)   # learn on augmented images
    eval_loss, eval_acc = evaluate(model, evaluation_loader)                 # same evaluation as before

    history_aug["train_loss"].append(train_loss)
    history_aug["train_accuracy"].append(train_acc)
    history_aug["evaluation_loss"].append(eval_loss)
    history_aug["evaluation_accuracy"].append(eval_acc)

    print(f"Epoch {epoch + 1}: train acc {train_acc:.3f}, eval acc {eval_acc:.3f}")

# Save only these 10 epochs
with open("history_augmented.json", "w") as f:
    json.dump(history_aug, f)

Epoch 1: train acc 0.605, eval acc 0.672
Epoch 2: train acc 0.647, eval acc 0.627
Epoch 3: train acc 0.680, eval acc 0.711
Epoch 4: train acc 0.675, eval acc 0.711
Epoch 5: train acc 0.714, eval acc 0.721
Epoch 6: train acc 0.709, eval acc 0.701
Epoch 7: train acc 0.700, eval acc 0.741
Epoch 8: train acc 0.729, eval acc 0.706
Epoch 9: train acc 0.756, eval acc 0.721
Epoch 10: train acc 0.705, eval acc 0.731


##### Question 5 — augmented evaluation loss

In [17]:
# Mean of evaluation_loss (augmented), rounded to 3 decimals
with open("history_augmented.json") as f:
    history_augmented = json.load(f)

print(round(np.mean(history_augmented["evaluation_loss"]), 3))

0.576


##### Question 6 — augmented evaluation accuracy

In [18]:
# Mean of the last five evaluation_accuracy values (epochs 6-10), rounded to 2 decimals
print(round(np.mean(history_augmented["evaluation_accuracy"][-5:]), 2))

0.72
